In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Merging RAW_recipes.csv with RAW_recipes_w_search_terms.csv

In [2]:
df_recipes = pd.read_csv("../data/RAW_recipes.csv")
df_recipes_w_search_terms = pd.read_csv("../data/recipes_w_search_terms.csv")

In [3]:
# Merge the original RAW_recipes dataset and recipes_w_search_terms dataset
new_columns = [col for col in df_recipes_w_search_terms.columns if col not in df_recipes.columns or col == 'id']
df_recipes = pd.merge(df_recipes, df_recipes_w_search_terms[new_columns], on='id', how='inner')

print("Recipes Dataset Info:\n")
df_recipes.info()
print("")
df_recipes.head()

Recipes Dataset Info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 222705 entries, 0 to 222704
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   name                 222704 non-null  object
 1   id                   222705 non-null  int64 
 2   minutes              222705 non-null  int64 
 3   contributor_id       222705 non-null  int64 
 4   submitted            222705 non-null  object
 5   tags                 222705 non-null  object
 6   nutrition            222705 non-null  object
 7   n_steps              222705 non-null  int64 
 8   steps                222705 non-null  object
 9   description          217940 non-null  object
 10  ingredients          222705 non-null  object
 11  n_ingredients        222705 non-null  int64 
 12  ingredients_raw_str  222705 non-null  object
 13  serving_size         222705 non-null  object
 14  servings             222705 non-null  int64 
 15  search_term

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,ingredients_raw_str,serving_size,servings,search_terms
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7,"[""1 lb winter squash (such as hubbard, ac...",1 (129 g),3,"{'mexican', 'side', 'vegetarian', 'baked'}"
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6,"[""1 (10 ounce) can prepared pizza crust (o...",1 (83 g),6,"{'breakfast', 'pizza', 'dinner'}"
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13,"[""2 lbs ground beef"",""1 1/2 medium ye...",1 (354 g),10,{'dinner'}
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11,"[""2 (4 ounce) packages alouette spreadable ...",1 (287 g),6,"{'side', 'dinner'}"
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8,"[""3 quarts tomato juice"",""1 pint app...",1 (2506 g),1,{'vegetarian'}


In [4]:
# Drop the data row with missing name (since there is only 1 such row)
df_recipes = df_recipes.dropna(subset=['name'])
print("Recipes Dataset Info:\n")
df_recipes.info()

Recipes Dataset Info:

<class 'pandas.core.frame.DataFrame'>
Index: 222704 entries, 0 to 222704
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   name                 222704 non-null  object
 1   id                   222704 non-null  int64 
 2   minutes              222704 non-null  int64 
 3   contributor_id       222704 non-null  int64 
 4   submitted            222704 non-null  object
 5   tags                 222704 non-null  object
 6   nutrition            222704 non-null  object
 7   n_steps              222704 non-null  int64 
 8   steps                222704 non-null  object
 9   description          217939 non-null  object
 10  ingredients          222704 non-null  object
 11  n_ingredients        222704 non-null  int64 
 12  ingredients_raw_str  222704 non-null  object
 13  serving_size         222704 non-null  object
 14  servings             222704 non-null  int64 
 15  search_terms    

In [5]:
# For data rows with missing description, we concatenate their recipe name and ingredients using '{name} is made using {ingredients}'
df_recipes['description'] = df_recipes.apply(
    lambda row: f"{row['name']} is made using {', '.join(eval(row['ingredients']))}" if pd.isnull(row['description']) else row['description'],
    axis=1
)

print("Recipes Dataset Info:\n")
df_recipes.info()
print("")
df_recipes.head()

Recipes Dataset Info:

<class 'pandas.core.frame.DataFrame'>
Index: 222704 entries, 0 to 222704
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   name                 222704 non-null  object
 1   id                   222704 non-null  int64 
 2   minutes              222704 non-null  int64 
 3   contributor_id       222704 non-null  int64 
 4   submitted            222704 non-null  object
 5   tags                 222704 non-null  object
 6   nutrition            222704 non-null  object
 7   n_steps              222704 non-null  int64 
 8   steps                222704 non-null  object
 9   description          222704 non-null  object
 10  ingredients          222704 non-null  object
 11  n_ingredients        222704 non-null  int64 
 12  ingredients_raw_str  222704 non-null  object
 13  serving_size         222704 non-null  object
 14  servings             222704 non-null  int64 
 15  search_terms    

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,ingredients_raw_str,serving_size,servings,search_terms
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7,"[""1 lb winter squash (such as hubbard, ac...",1 (129 g),3,"{'mexican', 'side', 'vegetarian', 'baked'}"
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6,"[""1 (10 ounce) can prepared pizza crust (o...",1 (83 g),6,"{'breakfast', 'pizza', 'dinner'}"
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13,"[""2 lbs ground beef"",""1 1/2 medium ye...",1 (354 g),10,{'dinner'}
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11,"[""2 (4 ounce) packages alouette spreadable ...",1 (287 g),6,"{'side', 'dinner'}"
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8,"[""3 quarts tomato juice"",""1 pint app...",1 (2506 g),1,{'vegetarian'}


In [6]:
# Convert 'submitted' to datetime
df_recipes['submitted'] = pd.to_datetime(df_recipes['submitted'])


In [7]:
# Take out Nutrition List Individual Values
df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)', 'protein (PDV)',
    'saturated fat (PDV)', 'carbohydrates (PDV)']] = df_recipes.nutrition.str.split(",", expand=True)

df_recipes['calories'] = df_recipes['calories'].str.replace('[', '')
df_recipes['carbohydrates (PDV)'] = df_recipes['carbohydrates (PDV)'].str.replace(']', '')
df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)', 'protein (PDV)', 'saturated fat (PDV)',
    'carbohydrates (PDV)']] = df_recipes[['calories', 'total fat (PDV)', 'sugar (PDV)', 'sodium (PDV)',
                                  'protein (PDV)', 'saturated fat (PDV)', 'carbohydrates (PDV)']].astype('float')

In [8]:
# Number of Rows with zero minutes
zero_minutes_count = df_recipes[df_recipes['minutes'] == 0].shape[0]
print(zero_minutes_count)

1000


In [9]:
# Drop rows where 'minutes' is 0
df_recipes = df_recipes[df_recipes['minutes'] != 0]
df_recipes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221704 entries, 0 to 222704
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   name                 221704 non-null  object        
 1   id                   221704 non-null  int64         
 2   minutes              221704 non-null  int64         
 3   contributor_id       221704 non-null  int64         
 4   submitted            221704 non-null  datetime64[ns]
 5   tags                 221704 non-null  object        
 6   nutrition            221704 non-null  object        
 7   n_steps              221704 non-null  int64         
 8   steps                221704 non-null  object        
 9   description          221704 non-null  object        
 10  ingredients          221704 non-null  object        
 11  n_ingredients        221704 non-null  int64         
 12  ingredients_raw_str  221704 non-null  object        
 13  serving_size       

In [10]:
# Number of Rows with zero n_steps
zero_n_steps_count = df_recipes[df_recipes['n_steps'] == 0].shape[0]
print(zero_n_steps_count)

1


In [11]:
# Drop rows where 'n_steps' is 0
df_recipes = df_recipes[df_recipes['n_steps'] != 0]
df_recipes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221703 entries, 0 to 222704
Data columns (total 23 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   name                 221703 non-null  object        
 1   id                   221703 non-null  int64         
 2   minutes              221703 non-null  int64         
 3   contributor_id       221703 non-null  int64         
 4   submitted            221703 non-null  datetime64[ns]
 5   tags                 221703 non-null  object        
 6   nutrition            221703 non-null  object        
 7   n_steps              221703 non-null  int64         
 8   steps                221703 non-null  object        
 9   description          221703 non-null  object        
 10  ingredients          221703 non-null  object        
 11  n_ingredients        221703 non-null  int64         
 12  ingredients_raw_str  221703 non-null  object        
 13  serving_size       

In [12]:
# Number of Rows with zero n_ingredients
zero_n_ingredients_count = df_recipes[df_recipes['n_ingredients'] == 0].shape[0]
print(zero_n_ingredients_count) # 0 so don't need to drop any rows

0


In [13]:
# Number of Rows with zero servings
zero_servings_count = df_recipes[df_recipes['servings'] == 0].shape[0]
print(zero_servings_count) # 0 so don't need to drop any rows

0


Observed that the data for `serving_size` is formatted by '1 (xxxx g)' for each data row. As such, I will be extracting only the serving size in grams and create a new column `serving_size_grams`.

In [14]:
# Extract the serving_size grams number into new column 'serving_size_grams'
df_recipes['serving_size_grams'] = df_recipes['serving_size'].str.extract(r'\((\d+)\s*g\)').astype(float)


In [15]:
# Check for any null values for serving_size_grams
null_serving_size_rows = df_recipes[df_recipes['serving_size_grams'].isnull()]
null_serving_size_rows # only 1 row

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,...,servings,search_terms,calories,total fat (PDV),sugar (PDV),sodium (PDV),protein (PDV),saturated fat (PDV),carbohydrates (PDV),serving_size_grams
163783,rainforest cafe rasta pasta for two or three,19962,20,10404,2002-02-18,"['30-minutes-or-less', 'time-to-make', 'course...","[529.4, 16.0, 19.0, 3.0, 39.0, 11.0, 30.0]",8,"['cook pasta according to package directions',...","fast, easy and can easily be doubled. makes t...",...,2,"{'dinner', 'pasta'}",529.4,16.0,19.0,3.0,39.0,11.0,30.0,NaN


In [16]:
# Find out the serving_size number for the data row containing the null value
null_serving_size_rows['serving_size']

163783    1 (-475 g)
Name: serving_size, dtype: object

In [17]:
# Fill with the found value
df_recipes['serving_size_grams'].fillna(475.0, inplace=True)

/var/folders/l2/51df6zgj7t9bq7c4lzw3dlcr0000gn/T/ipykernel_20260/738133672.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_recipes['serving_size_grams'].fillna(475.0, inplace=True)


In [18]:
# Number of Rows with zero serving_size_grams
zero_serving_size_grams_count = df_recipes[df_recipes['serving_size_grams'] == 0].shape[0]
print(zero_serving_size_grams_count)

70


In [19]:
# Drop rows where 'serving_size_grams' is 0
df_recipes = df_recipes[df_recipes['serving_size_grams'] != 0]
df_recipes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221633 entries, 0 to 222704
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   name                 221633 non-null  object        
 1   id                   221633 non-null  int64         
 2   minutes              221633 non-null  int64         
 3   contributor_id       221633 non-null  int64         
 4   submitted            221633 non-null  datetime64[ns]
 5   tags                 221633 non-null  object        
 6   nutrition            221633 non-null  object        
 7   n_steps              221633 non-null  int64         
 8   steps                221633 non-null  object        
 9   description          221633 non-null  object        
 10  ingredients          221633 non-null  object        
 11  n_ingredients        221633 non-null  int64         
 12  ingredients_raw_str  221633 non-null  object        
 13  serving_size       

In [20]:
# Drop not needed columns
df_recipes = df_recipes.drop(columns=['serving_size','nutrition'])
df_recipes.info()

<class 'pandas.core.frame.DataFrame'>
Index: 221633 entries, 0 to 222704
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   name                 221633 non-null  object        
 1   id                   221633 non-null  int64         
 2   minutes              221633 non-null  int64         
 3   contributor_id       221633 non-null  int64         
 4   submitted            221633 non-null  datetime64[ns]
 5   tags                 221633 non-null  object        
 6   n_steps              221633 non-null  int64         
 7   steps                221633 non-null  object        
 8   description          221633 non-null  object        
 9   ingredients          221633 non-null  object        
 10  n_ingredients        221633 non-null  int64         
 11  ingredients_raw_str  221633 non-null  object        
 12  servings             221633 non-null  int64         
 13  search_terms       

In [21]:
# Save to data folder
df_recipes.to_csv("../data/df_recipes_merged.csv", index=False)

## Merge df_interactions_with_ratings.csv and df_recipes_merged.csv




In [22]:
df_recipes_merged = pd.read_csv("../data/df_recipes_merged.csv")

In [23]:
df_recipes_merged.head()
df_recipes_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 221633 entries, 0 to 221632
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   name                 221633 non-null  object 
 1   id                   221633 non-null  int64  
 2   minutes              221633 non-null  int64  
 3   contributor_id       221633 non-null  int64  
 4   submitted            221633 non-null  object 
 5   tags                 221633 non-null  object 
 6   n_steps              221633 non-null  int64  
 7   steps                221633 non-null  object 
 8   description          221633 non-null  object 
 9   ingredients          221633 non-null  object 
 10  n_ingredients        221633 non-null  int64  
 11  ingredients_raw_str  221633 non-null  object 
 12  servings             221633 non-null  int64  
 13  search_terms         221633 non-null  object 
 14  calories             221633 non-null  float64
 15  total fat (PDV)  

In [24]:
df_interactions = pd.read_csv("../data/df_interactions_with_ratings.csv")

In [25]:
df_interactions.head()

,user_id,recipe_id,date,rating,review,weighted_score
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...,4.6
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall...",5.0
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...,4.6
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...,4.4
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin...",3.8


In [26]:
df_aggregated = df_interactions.merge(df_recipes_merged, left_on="recipe_id", right_on="id", how='left')
df_aggregated.head()

,user_id,recipe_id,date,rating,review,weighted_score,name,id,minutes,contributor_id,...,servings,search_terms,calories,total fat (PDV),sugar (PDV),sodium (PDV),protein (PDV),saturated fat (PDV),carbohydrates (PDV),serving_size_grams
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...,4.6,white bean green chile pepper soup,40893.0,495.0,1533.0,...,5.0,{'soup'},204.8,5.0,9.0,26.0,24.0,2.0,10.0,292.0
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall...",5.0,white bean green chile pepper soup,40893.0,495.0,1533.0,...,5.0,{'soup'},204.8,5.0,9.0,26.0,24.0,2.0,10.0,292.0
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...,4.6,devilicious cookie cake delights,44394.0,20.0,56824.0,...,36.0,"{'cookie', 'dessert', 'cake', 'lunch'}",132.3,11.0,39.0,5.0,4.0,11.0,5.0,28.0
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...,4.4,baked potato toppings,85009.0,10.0,64342.0,...,1.0,{'baked'},2786.2,342.0,134.0,290.0,161.0,301.0,42.0,1177.0
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin...",3.8,baked potato toppings,85009.0,10.0,64342.0,...,1.0,{'baked'},2786.2,342.0,134.0,290.0,161.0,301.0,42.0,1177.0


In [27]:
df_aggregated.drop(columns=["id"], inplace=True)

In [28]:
df_aggregated.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1132367 entries, 0 to 1132366
Data columns (total 27 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   user_id              1132367 non-null  int64  
 1   recipe_id            1132367 non-null  int64  
 2   date                 1132367 non-null  object 
 3   rating               1132367 non-null  int64  
 4   review               1132198 non-null  object 
 5   weighted_score       1132367 non-null  float64
 6   name                 1088367 non-null  object 
 7   minutes              1088367 non-null  float64
 8   contributor_id       1088367 non-null  float64
 9   submitted            1088367 non-null  object 
 10  tags                 1088367 non-null  object 
 11  n_steps              1088367 non-null  float64
 12  steps                1088367 non-null  object 
 13  description          1088367 non-null  object 
 14  ingredients          1088367 non-null  object 
 15

In [29]:
# Some user interaction rows do not have recipes merged to them
unmapped_recipe_ids = set(df_interactions['recipe_id']) - set(df_recipes_merged['id'])
print(len(unmapped_recipe_ids))
unmapped_recipe_ids_list = list(unmapped_recipe_ids)
print(unmapped_recipe_ids_list[:50])

10004
[262145, 98318, 262160, 327696, 294933, 360477, 229414, 262186, 491565, 458798, 393265, 98354, 65588, 262200, 229438, 262210, 65606, 393286, 393288, 360521, 491594, 32845, 294991, 196693, 360534, 295001, 393306, 65628, 458849, 229474, 327778, 196708, 295018, 426094, 98415, 491631, 458868, 426105, 393346, 163973, 163978, 229520, 327826, 458902, 98459, 196768, 32931, 295079, 295080, 65705]


In [30]:
# Drop the rows
df_aggregated_cleaned = df_aggregated.dropna(subset=df_aggregated.columns[6:])
df_aggregated_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1088367 entries, 0 to 1132366
Data columns (total 27 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   user_id              1088367 non-null  int64  
 1   recipe_id            1088367 non-null  int64  
 2   date                 1088367 non-null  object 
 3   rating               1088367 non-null  int64  
 4   review               1088206 non-null  object 
 5   weighted_score       1088367 non-null  float64
 6   name                 1088367 non-null  object 
 7   minutes              1088367 non-null  float64
 8   contributor_id       1088367 non-null  float64
 9   submitted            1088367 non-null  object 
 10  tags                 1088367 non-null  object 
 11  n_steps              1088367 non-null  float64
 12  steps                1088367 non-null  object 
 13  description          1088367 non-null  object 
 14  ingredients          1088367 non-null  object 
 15  n_i

In [31]:
# Convert 'date' to datetime
df_aggregated_cleaned['date'] = pd.to_datetime(df_aggregated_cleaned['date'])

/var/folders/l2/51df6zgj7t9bq7c4lzw3dlcr0000gn/T/ipykernel_20260/4197313154.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aggregated_cleaned['date'] = pd.to_datetime(df_aggregated_cleaned['date'])


In [32]:
# Save to data folder
df_aggregated_cleaned.to_csv("../data/df_agg_cleaned.csv", index=False)

In [33]:
df_aggregated_cleaned.head()

,user_id,recipe_id,date,rating,review,weighted_score,name,minutes,contributor_id,submitted,...,servings,search_terms,calories,total fat (PDV),sugar (PDV),sodium (PDV),protein (PDV),saturated fat (PDV),carbohydrates (PDV),serving_size_grams
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for...,4.6,white bean green chile pepper soup,495.0,1533.0,2002-09-21,...,5.0,{'soup'},204.8,5.0,9.0,26.0,24.0,2.0,10.0,292.0
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall...",5.0,white bean green chile pepper soup,495.0,1533.0,2002-09-21,...,5.0,{'soup'},204.8,5.0,9.0,26.0,24.0,2.0,10.0,292.0
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not...,4.6,devilicious cookie cake delights,20.0,56824.0,2002-10-27,...,36.0,"{'cookie', 'dessert', 'cake', 'lunch'}",132.3,11.0,39.0,5.0,4.0,11.0,5.0,28.0
3,126440,85009,2010-02-27,5,I made the Mexican topping and took it to bunk...,4.4,baked potato toppings,10.0,64342.0,2004-02-25,...,1.0,{'baked'},2786.2,342.0,134.0,290.0,161.0,301.0,42.0,1177.0
4,57222,85009,2011-10-01,5,"Made the cheddar bacon topping, adding a sprin...",3.8,baked potato toppings,10.0,64342.0,2004-02-25,...,1.0,{'baked'},2786.2,342.0,134.0,290.0,161.0,301.0,42.0,1177.0
